In [1]:
import pandas as pd
import numpy as np
from cdc_ml.config import POLLS_PROCESSED,CUSTOMER_CLASS_PROCESSED
import xgboost as xgb
from cdc_ml.modeling.train import train
from cdc_ml.features.build_features import assign_class_type
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
import tqdm
from loguru import logger

2026-05-23 17:43:50.228 | INFO     | cdc_ml.config:<module>:12 - PROJ_ROOT path is: C:\Users\zhiju\Desktop\cdc_ml


In [2]:
df = pd.read_parquet(POLLS_PROCESSED)
df_class = pd.read_parquet(CUSTOMER_CLASS_PROCESSED)

In [3]:
df= assign_class_type(df,df_class)

In [4]:
df = df.loc[~(df["username"]=="anmol")]

In [5]:
df.sort_values(by='cycle_start').tail()

,id,username,cycle_start,cycle_end,polling_at,booking_hour,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,polling_month,polling_day,polling_dow,polling_hour,hours_into_cycle,class_type,is_one_team
2047,2047,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-24 14:00:00+08:00,NaT,False,4,21,1,0,4,24,4,14,86.0,0,1
2048,2048,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-24 15:00:00+08:00,NaT,False,4,21,1,0,4,24,4,15,87.0,0,1
2049,2049,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-24 16:00:00+08:00,NaT,False,4,21,1,0,4,24,4,16,88.0,0,1
2021,2021,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-23 12:00:00+08:00,NaT,False,4,21,1,0,4,23,3,12,60.0,0,1
1961,1961,ali,2026-04-21 00:00:00+08:00,2026-04-25 23:59:00+08:00,2026-04-21 00:00:00+08:00,NaT,False,4,21,1,0,4,21,1,0,0.0,0,1


In [6]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr, te = next(sgkf.split(df, y=df["has_booking"], groups=df["username"]))
df_train, df_test = df.iloc[tr], df.iloc[te]

In [7]:
df_train

,id,username,cycle_start,cycle_end,polling_at,booking_hour,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,polling_month,polling_day,polling_dow,polling_hour,hours_into_cycle,class_type,is_one_team
0,0,addity,2025-08-13 21:00:00+08:00,2025-08-28 11:36:00+08:00,2025-08-13 21:00:00+08:00,NaT,False,8,13,2,21,8,13,2,21,0.00,1,1
1,1,addity,2025-08-13 21:00:00+08:00,2025-08-28 11:36:00+08:00,2025-08-13 22:00:00+08:00,NaT,False,8,13,2,21,8,13,2,22,1.00,1,1
2,2,addity,2025-08-13 21:00:00+08:00,2025-08-28 11:36:00+08:00,2025-08-13 23:00:00+08:00,NaT,False,8,13,2,21,8,13,2,23,2.00,1,1
3,3,addity,2025-08-13 21:00:00+08:00,2025-08-28 11:36:00+08:00,2025-08-14 00:00:00+08:00,NaT,False,8,13,2,21,8,14,3,0,3.00,1,1
4,4,addity,2025-08-13 21:00:00+08:00,2025-08-28 11:36:00+08:00,2025-08-14 01:00:00+08:00,NaT,False,8,13,2,21,8,14,3,1,4.00,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31825,31528,srpr,2026-02-17 13:03:00+08:00,2026-03-02 12:00:00+08:00,2026-03-02 07:00:00+08:00,NaT,False,2,17,1,13,3,2,0,7,305.95,0,0
31826,31529,srpr,2026-02-17 13:03:00+08:00,2026-03-02 12:00:00+08:00,2026-03-02 08:00:00+08:00,NaT,False,2,17,1,13,3,2,0,8,306.95,0,0
31827,31530,srpr,2026-02-17 13:03:00+08:00,2026-03-02 12:00:00+08:00,2026-03-02 09:00:00+08:00,NaT,False,2,17,1,13,3,2,0,9,307.95,0,0
31828,31531,srpr,2026-02-17 13:03:00+08:00,2026-03-02 12:00:00+08:00,2026-03-02 10:00:00+08:00,NaT,False,2,17,1,13,3,2,0,10,308.95,0,0


In [8]:
print(len(df_train)/len(df))
print(len(df_test)/len(df))

0.8152519291874716
0.18474807081252836


In [9]:
df_train["username"].value_counts(normalize=True).sort_values()

username
ajithak    0.002028
max        0.002028
isyaf      0.002903
bhara      0.005727
pakning    0.006005
sara       0.007119
carol      0.007636
phuc       0.008312
nur        0.008630
aswath     0.008909
d          0.010500
bryan      0.014119
np         0.016903
addity     0.019090
bw         0.019130
fir        0.020721
natar      0.021954
lucinda    0.023624
gohguan    0.028078
apple      0.031300
jun        0.033408
joy        0.033487
brendon    0.041521
faith      0.045379
srpr       0.059935
ali        0.061645
matt       0.062599
mya        0.065861
anaya      0.068804
ryan       0.078229
flower     0.090200
jy         0.094217
Name: proportion, dtype: float64

In [10]:
df_test["username"].value_counts(normalize=True).sort_values()

username
ranjith    0.006318
poopie     0.133731
tomato     0.189365
kim        0.670586
Name: proportion, dtype: float64

In [11]:
df_train["has_booking"].mean()

np.float64(0.013124403436207445)

In [12]:
df_test["has_booking"].mean()

np.float64(0.014566514566514567)

In [13]:
#seed = np.random.randint(0,100)
seed = 42
# if the features below is not removed , the model will overfit 
oof_xgb,oof_marg,oof_const,models=train(df_train,["cycle_start_month","cycle_start_day","cycle_start_dow","cycle_start_hour","polling_month","polling_day","hours_into_cycle","class_type","is_one_team"],seed)

      polling_dow  polling_hour
6348            2            19


[0. 0.]
fold 0: train n= 20015 (2025-08-12 → 2026-04-21)  val n=  5129 (2025-08-19 → 2026-02-11)  
  train_pos=0.013  val_pos=0.013  
  marg_dow_brier=0.0127  marg_dow_pr=0.0174  
  marg_hour_brier=0.0126  marg_hour_pr=0.0291  
  add_brier=0.0125  add_pr=0.0370  
  joint_brier=0.0127  joint_pr=0.0314  
  xgb_brier_val=0.0125  xgb_pr_val=0.0363  
  xgb_brier_tr=0.0128  xgb_pr_tr=0.0389  

[0. 0.]
fold 1: train n= 19984 (2025-08-12 → 2026-04-21)  val n=  5160 (2025-08-22 → 2026-04-01)  
  train_pos=0.013  val_pos=0.013  
  marg_dow_brier=0.0130  marg_dow_pr=0.0143  
  marg_hour_brier=0.0128  marg_hour_pr=0.0324  
  add_brier=0.0128  add_pr=0.0393  
  joint_brier=0.0129  joint_pr=0.0378  
  xgb_brier_val=0.0128  xgb_pr_val=0.0414  
  xgb_brier_tr=0.0128  xgb_pr_tr=0.0356  

[0. 0.]
fold 2: train n= 19954 (2025-08-12 → 2026-04-01)  val n=  5190 (2025-08-13 → 2026-04-21)  
  train_pos=0.013  val_pos=0.013  
  marg_dow_brier=0.0124  marg_dow_pr=0.0155  
  marg_hour_brier=0.0124  marg_hour_pr

In [14]:
from sklearn.inspection import permutation_importance


In [15]:
# from pathlib import Path
# from sklearn.model_selection import permutation_test_score
# from cdc_ml.features.build_features import drop_meta_high_card_cols
# from cdc_ml.modeling.train import time_ordered_kfold
# from cdc_ml.config import STAGE_1_PROCESSED

# def perm(
#     data_path: Path = STAGE_1_PROCESSED,
#     n_permutations: int = 1000,
#     seed: int = 42,
# ):
#     df = pd.read_parquet(data_path)
#     df_train, _ = temporal_split(df)

#     y = df_train["has_booking"].to_numpy()
#     X = drop_meta_high_card_cols(df_train).drop(columns=["has_booking"]+["cycle_start_month","cycle_start_day","cycle_start_dow","cycle_start_hour","polling_month","polling_day","hours_into_cycle","class_type","is_one_team"])

#     splits = list(time_ordered_kfold(df_train, time_col="cycle_start"))

#     clf = xgb.XGBClassifier(
#         n_estimators=500,
#         objective="binary:logistic",
#         eval_metric="aucpr",
#         learning_rate=0.03,
#         max_depth=1,
#         min_child_weight=10,
#         subsample=0.8,
#         colsample_bytree=0.8,
#         reg_lambda=1.0,
#         tree_method="hist",
#         device="cuda",
#         random_state=seed,
#         n_jobs=1,                  # let permutation_test_score parallelize
#     )

#     score, perm_scores, pvalue = permutation_test_score(
#         clf, X, y,
#         scoring="average_precision",
#         cv=splits,                 # pre-computed list of (train, val) index pairs
#         n_permutations=n_permutations,
#         n_jobs=-1,
#         random_state=seed,
#         verbose=1,
#     )

#     print(
#         f"\ntrue PR-AUC = {score:.4f}\n"
#         f"null        = {perm_scores.mean():.4f} ± {perm_scores.std():.4f}\n"
#         f"p-value     = {pvalue:.4f}  (n_perm={n_permutations})"
#     )

# perm(STAGE_1_PROCESSED)